# 기본 BM25와 Kiwi BM25 비교

BGE-m3-ko 임베딩, 같은 질문에서 기본·Kiwi BM25의 검색과 답변을 비교한 버전.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


## 준비. 패키지 설치와 드라이브 연결


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" accelerate rank_bm25 kiwipiepy
%pip install -U "ragas==0.3.9" "langchain-openai"


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()


df = pd.read_csv(metadata_csv, encoding="utf-8")
loaded_files = []

for name in sorted_file_names:
    full_path = os.path.join(folder, name)

    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = loader.load()[0].page_content

    loaded_files.append((name, text))

print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]

documents = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        continue

    row = result_df.iloc[0]
    metadata = {column: row[column] for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))

print("메타데이터와 연결한 문서:", len(documents))


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 모델 준비

사업별 FAISS 인덱스는 해당 사업의 질문이 처음 들어올 때 만듭니다.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model_name = "dragonkue/BGE-m3-ko" #"dragonkue/BGE-m3-ko" #"nlpai-lab/KURE-v1" #"BAAI/bge-m3" # "Alibaba-NLP/gte-multilingual-base"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

print("임베딩 모델:", embedding_model_name)


## 6. 질문에서 사업 식별 후 하이브리드 검색

질문에 적힌 정식 사업명을 문서 메타데이터의 사업명 목록과 비교합니다. 띄어쓰기와 문장부호 차이는 무시합니다. 줄임말은 `PROJECT_ALIASES`에 등록하세요. 사업을 하나로 특정할 수 없으면 오류를 내며, 평가 데이터의 `project` 필드는 검색에 사용하지 않습니다.


In [ ]:
from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def identify_project(query):
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
        key for key in project_catalog
        if normalize_project_text(key[1]) in normalized_query
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )


from kiwipiepy import Kiwi

kiwi = Kiwi()
BM25_MODES = ("current", "kiwi")
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """Tokenize documents and queries identically for Kiwi BM25."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]


dense_retrievers_by_project = {}
bm25_retrievers_by_project_and_mode = {}


def get_retrievers(project_key, bm25_mode):
    if bm25_mode not in BM25_MODES:
        raise ValueError(f"지원하지 않는 BM25 방식: {bm25_mode}")

    project_chunks = project_chunks_by_key[project_key]
    if project_key not in dense_retrievers_by_project:
        store = FAISS.from_documents(project_chunks, embeddings)
        dense_retrievers_by_project[project_key] = store.as_retriever(search_kwargs={"k": 20})

    bm25_key = (project_key, bm25_mode)
    if bm25_key not in bm25_retrievers_by_project_and_mode:
        if bm25_mode == "kiwi":
            bm25 = BM25Retriever.from_documents(
                project_chunks, preprocess_func=kiwi_morphs
            )
        else:
            bm25 = BM25Retriever.from_documents(project_chunks)
        bm25.k = 20
        bm25_retrievers_by_project_and_mode[bm25_key] = bm25

    return (
        dense_retrievers_by_project[project_key],
        bm25_retrievers_by_project_and_mode[bm25_key],
    )


def rrf_search(query, project_key, bm25_mode, k=60):
    dense_retriever, bm25_retriever = get_retrievers(project_key, bm25_mode)
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 7. HyDE로 검색 문장 만들기


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

In [ ]:
# 이 비교에서는 아래 셀의 GPT-5-mini를 사용합니다.
# 기존 EXAONE AWQ 로딩은 실행하지 않습니다.
method = "basic"


In [ ]:
import os
from langchain_openai import ChatOpenAI

method = "basic"  # "basic" 또는 "hyde"

# Colab 보안 비밀에 키를 등록한 경우
if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

chat_model = ChatOpenAI(model="gpt-5-mini-2025-08-07")

In [ ]:
def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")

## 8. 질문에서 식별한 사업 안에서 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device="cuda",
)

# 질문과 검색 설정이 같을 때만 검색 결과를 재사용합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method, bm25_mode="current"):
    project_key = identify_project(query)
    cache_key = (project_key, method, query, bm25_mode)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text, project_key, bm25_mode)
        ranking = reranker.rank(
            query,
            [doc.page_content for doc in found],
            top_k=6,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (
            project_key, search_text, found, reranked_docs
        )
    return retrieval_cache[cache_key]


## 9. 일반 질문 함수

아래 함수 셀까지 실행하면 `answer_question("정식 사업명의 공고기간은?")`처럼 평가셋 밖의 질문에도 사용할 수 있습니다. 정식 명칭을 줄여 부르려면 검색 셀의 `PROJECT_ALIASES`에 등록하세요.


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''


def answer_question(query, search_method=None, bm25_mode="current"):
    selected_method = method if search_method is None else search_method
    project_key, search_text, found, reranked_docs = retrieve_and_rerank(
        query, selected_method, bm25_mode
    )
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    answer = chat_model.invoke(
        answer_rules.format(context=context, query=query)
    ).content.strip()
    return {
        "query": query,
        "bm25_mode": bm25_mode,
        "identified_agency": project_key[0],
        "identified_project": project_key[1],
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
    }



### 평가 질문 불러오기 (선택)

평가셋을 사용할 때만 실행합니다. 일반 질문의 사업 식별에는 평가 데이터의 `project` 값을 사용하지 않습니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((path for path in eval_dataset_candidates if path.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError(
        "rag_eval_dataset.json을 Google Drive 내 드라이브 루트 또는 노트북 폴더에 두세요."
    )

with eval_dataset_path.open(encoding="utf-8") as file:
    all_eval_records = json.load(file)

EVAL_PROJECT_COUNT = 2
EVAL_QUESTIONS_PER_PROJECT = 2
available_projects = list(dict.fromkeys(record["project"] for record in all_eval_records))
selected_projects = available_projects[:EVAL_PROJECT_COUNT]

selected_eval_records = []
selected_count = Counter()
for record in all_eval_records:
    project = record["project"]
    if project in selected_projects and selected_count[project] < EVAL_QUESTIONS_PER_PROJECT:
        selected_eval_records.append(record)
        selected_count[project] += 1

if not selected_eval_records:
    raise ValueError("선택된 평가 문항이 없습니다. 사업명 필터와 입력 파일을 확인하세요.")

evaluation_data = []
for record in selected_eval_records:
    if not record.get("id") or not record.get("question") or not record.get("reference"):
        raise ValueError(f"필수 평가 필드가 비어 있습니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"],
        "query": record["question"],
        "reference": record["reference"],
        "project": record["project"],
        "question_type": record["question_type"],
    })

if len({item["id"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 문항 ID가 중복됩니다.")
if len({item["query"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 질문이 중복됩니다.")

question = evaluation_data  # 기존 검색·답변 생성 반복문에서 사용
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("유형별:", dict(Counter(item["question_type"] for item in evaluation_data)))
print("사업별:", dict(Counter(item["project"] for item in evaluation_data)))


### 현재 BM25와 Kiwi BM25를 같은 질문에서 비교

기본 설정은 두 사업에서 각 2문항씩, 총 4문항입니다. 사업 수와 문항 수는 위 셀의
`EVAL_PROJECT_COUNT`, `EVAL_QUESTIONS_PER_PROJECT`로 바꿀 수 있습니다.
두 방식은 같은 문서 청크, 한국어 BGE-M3 임베딩, dense 검색, RRF, 재정렬 모델,
최종 문맥 6개, GPT-5-mini 답변 생성을 사용합니다. BM25의 토큰화만 다릅니다.


In [ ]:
import pandas as pd

answers_by_mode = {mode: [] for mode in BM25_MODES}
comparison_rows = []

for item in evaluation_data:
    paired = {}
    for bm25_mode in BM25_MODES:
        result = answer_question(item["query"], bm25_mode=bm25_mode)
        answers_by_mode[bm25_mode].append(result)
        paired[bm25_mode] = result

    current = paired["current"]
    kiwi_result = paired["kiwi"]
    if current["identified_project"] != kiwi_result["identified_project"]:
        raise RuntimeError(f'{item["id"]}: 두 방식의 검색 사업이 다릅니다.')

    comparison_rows.append({
        "id": item["id"],
        "project": item["project"],
        "question_type": item["question_type"],
        "question": item["query"],
        "identified_project": current["identified_project"],
        "current_chunk_ids": json.dumps(current["chunk_ids"], ensure_ascii=False),
        "kiwi_chunk_ids": json.dumps(kiwi_result["chunk_ids"], ensure_ascii=False),
        "shared_chunk_count": len(set(current["chunk_ids"]) & set(kiwi_result["chunk_ids"])),
        "current_answer": current["answer"],
        "kiwi_answer": kiwi_result["answer"],
        "current_contexts": json.dumps(current["contexts"], ensure_ascii=False),
        "kiwi_contexts": json.dumps(kiwi_result["contexts"], ensure_ascii=False),
        "reference": item["reference"],
    })
    print(f'\n{item["id"]}: {item["query"]}')
    print("사업:", current["identified_project"])
    print("현재 청크:", current["chunk_ids"])
    print("Kiwi 청크:", kiwi_result["chunk_ids"])
    print("현재 답변:", current["answer"])
    print("Kiwi 답변:", kiwi_result["answer"])

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = eval_dataset_path.with_name("rag_comparison_current_vs_kiwi.csv")
comparison_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
display(comparison_df[["id", "project", "current_chunk_ids", "kiwi_chunk_ids", "shared_chunk_count"]])
print("비교 CSV:", comparison_path)

# Older single-mode cells can still inspect the current branch if needed.
answers = answers_by_mode["current"]


## RAGAS 점수 비교

동일한 평가 질문과 기준 답변으로 두 방식을 각각 채점합니다. 생성과 채점에 API 비용이
발생하며, 여기서는 4문항 × 2방식만 실행합니다. 검색 문맥과 답변의 실제 차이도
앞 셀의 CSV에서 함께 확인하세요.


In [ ]:
import os
import sys
import types

module_name = "langchain_community.chat_models.vertexai"
if module_name not in sys.modules:
    module = types.ModuleType(module_name)
    module.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules[module_name] = module

from langchain_openai import ChatOpenAI
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    FactualCorrectness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)
from ragas.run_config import RunConfig

if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

N = len(evaluation_data)
if any(len(answers_by_mode[mode]) != N for mode in BM25_MODES):
    raise ValueError("두 방식의 답변을 모두 생성한 뒤 평가를 실행하세요.")

judge = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-5-mini-2025-08-07"),
    bypass_temperature=True,
    bypass_n=True,
)
metrics = [
    Faithfulness(),
    FactualCorrectness(mode="f1"),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]
score_columns = [
    "faithfulness",
    "factual_correctness(mode=f1)",
    "llm_context_precision_with_reference",
    "context_recall",
]

score_frames = []
for bm25_mode in BM25_MODES:
    mode_answers = answers_by_mode[bm25_mode]
    samples = []
    for item, answer_record in zip(evaluation_data, mode_answers):
        if item["query"] != answer_record["query"]:
            raise ValueError(f'{item["id"]}: 질문과 답변 순서가 다릅니다.')
        samples.append({
            "user_input": item["query"],
            "retrieved_contexts": answer_record["contexts"],
            "response": answer_record["answer"].split("답변:", 1)[-1].strip(),
            "reference": item["reference"],
        })

    result = evaluate(
        dataset=EvaluationDataset.from_list(samples),
        metrics=metrics,
        llm=judge,
        run_config=RunConfig(max_workers=1, max_retries=1, timeout=600),
        raise_exceptions=True,
    )
    frame = result.to_pandas()
    frame.insert(0, "id", [item["id"] for item in evaluation_data])
    frame.insert(1, "project", [item["project"] for item in evaluation_data])
    frame.insert(2, "bm25_mode", bm25_mode)
    if frame[score_columns].isna().any().any():
        raise RuntimeError(f"{bm25_mode}: 평가 점수가 누락됐습니다.")
    score_frames.append(frame)

score_df = pd.concat(score_frames, ignore_index=True)
score_path = eval_dataset_path.with_name("ragas_current_vs_kiwi.csv")
score_df.to_csv(score_path, index=False, encoding="utf-8-sig")
display(score_df[["id", "project", "bm25_mode", *score_columns]])
display(score_df.groupby("bm25_mode")[score_columns].mean())
print("점수 CSV:", score_path)


In [ ]:
import re
import unicodedata


def compact(text):
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(text)))


with eval_dataset_path.open(encoding="utf-8") as file:
    original_records = json.load(file)
records_by_id = {record["id"]: record for record in original_records}


def extract_quotes(evidence_quotes):
    quotes = []
    seen = set()
    for evidence in evidence_quotes or []:
        for curved, straight in re.findall(r'[“]([^”]+)[”]|"([^"]+)"', str(evidence)):
            quote = curved or straight
            if "…" in quote or "..." in quote:
                continue
            normalized = compact(quote)
            if len(normalized) >= 8 and re.search(r"[가-힣A-Za-z]", normalized) and normalized not in seen:
                seen.add(normalized)
                quotes.append(normalized)
    return quotes


coverage_rows = []
for bm25_mode in BM25_MODES:
    for item, answer_record in zip(evaluation_data, answers_by_mode[bm25_mode]):
        original = records_by_id[item["id"]]
        quotes = extract_quotes(original.get("evidence_quotes", []))
        contexts = [compact(context) for context in answer_record["contexts"]]
        found = sum(any(quote in context for context in contexts) for quote in quotes)
        coverage_rows.append({
            "id": item["id"],
            "project": item["project"],
            "bm25_mode": bm25_mode,
            "evidence_quote_count": len(quotes),
            "found_quote_count": found,
            "quote_coverage": found / len(quotes) if quotes else None,
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_path = eval_dataset_path.with_name("evidence_current_vs_kiwi.csv")
coverage_df.to_csv(coverage_path, index=False, encoding="utf-8-sig")
display(coverage_df)
print("근거 포함률 CSV:", coverage_path)
